# OpenAPI 를 이용한 데이터 수집 

## API(Application Progmaming Interface)
- 프로그램과 프로그램이 서로 데이터를 주고 받기 위한 규칙

## OpenAPI
- 외부 개발자가 사용할 수 있도록 공개된 API
- HTTP 기반 요청
- JSON or XML 형태의 응답
- API Key 필요

## 환경변수 관리
- 애플리케이션의 Client ID, Client Sercret 값은 절대 외부에 노출 되어서는 안된다.
- dotenv 를 통해 관리할 수 있다.

In [ ]:
# 셀 매직 명령어(%)로 python-dotenv 설치
%pip install python-dotenv

In [ ]:
# .env의 내용을 읽어서 환경 변수로 등록 처리
from dotenv import load_dotenv
load_dotenv()  # true가 나오면 .env 파일이 정상 로드 된 것

In [1]:
# 환경 변수로 등록 된 클라이언트 아이디와 시크릿 키를 읽어서 변수에 저장
import os

NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

## API 요청
- 네이버 Documents 에서 API의 요청, 응답 규칙 확인 후 예제 수행 (https://developers.naver.com/docs/serviceapi/search/blog/blog.md#python)

In [ ]:
# 네이버 검색 API 예제 - 뉴스 검색
import os
import sys
import urllib.request
client_id = NAVER_CLIENT_ID
client_secret = NAVER_CLIENT_SECRET
encText = urllib.parse.quote("인공지능")
url = "https://openapi.naver.com/v1/search/news?start=20&display=20&query=" + encText # JSON 결과, 요청 파라미터: display=20 (기본값:10, 최대값:100)
# url = "https://openapi.naver.com/v1/search/news.xml?query=" + encText # XML 결과
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)

In [ ]:
## request 모듈을 사용한 요청으로 변경
%pip install requests

In [ ]:
# requests 모듈
import requests
import csv

url = 'https://openapi.naver.com/v1/search/news.json'

headers = {
    "X-Naver-Client-Id" : NAVER_CLIENT_ID,
    "X-Naver-Client-Secret" : NAVER_CLIENT_SECRET
}

# query string
params = {
    'query' : '인공지능',  # 검색어
    'display' : 10,      # 10 ~ 100 
    'start' : 1,         # 1 ~ 100 
    'sort' : 'sim'       # sim or date
}

# 요청
response = requests.get(url, headers=headers, params=params)
print(f"HTTP Status Code: {response.status_code}")  # 200: 성공

# 결과 출력
if response.status_code == 200:
    # print(response.text)
    # response 객체는 json 문자열 형태로 응답이 돌아왔으므로 json() 메소드를 통해 dict로 변환
    data = response.json()
    print(type(data))
    
    # csv 파일로 저장하기
    with open('naver_news_인공지능.csv', 'w', encoding='utf-8', newline='') as f:
        # items 리스트의 첫 번째 요소의 키를 필드명으로 사용
        fieldnames = list(data['items'][0].keys())
        # csv 파일에 저장할 필드명 정의
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()                  # 헤더 작성
        writer.writerows(data['items'])      # 데이터 작성
else:
    print(f"Error Code: {response.status_code}")

## json 데이터 형식을 xml 데이터 형식 응답으로 변경

In [ ]:
# requests 모듈
import requests
import csv

url = 'https://openapi.naver.com/v1/search/news.xml'

headers = {
    "X-Naver-Client-Id" : NAVER_CLIENT_ID,
    "X-Naver-Client-Secret" : NAVER_CLIENT_SECRET
}

# query string
params = {
    'query' : '인공지능',  # 검색어
    'display' : 10,      # 10 ~ 100 
    'start' : 1,         # 1 ~ 100 
    'sort' : 'sim'       # sim or date
}

# 요청
response = requests.get(url, headers=headers, params=params)
print(f"HTTP Status Code: {response.status_code}")  # 200: 성공

# 결과 출력
if response.status_code == 200:
    print(response.text)
    
    # XML 파일로 저장 - response.content는 bytes 타입이므로 바이너리 모드로 파일을 열어서 저장한다.
    with open('naver_news_인공지능.xml', 'wb') as f:
        f.write(response.content)
        # xml extension 설치 후 option + shift + f : xml 형식이 보기좋게 바뀐다.
    
else:
    print(f"Error Code: {response.status_code}")

HTTP Status Code: 200
<?xml version="1.0" encoding="UTF-8"?><rss version="2.0"><channel><title>Naver Open API - news ::&apos;인공지능&apos;</title><link>https://search.naver.com</link><description>Naver Search Result</description><lastBuildDate>Fri, 13 Mar 2026 10:43:04 +0900</lastBuildDate><total>3767386</total><start>1</start><display>10</display><item><title>과기정통·산업·중기부, 4320억 규모 AI전환 사업 통합 공고</title><originallink>https://www.newsis.com/view/NISX20260312_0003545057</originallink><link>https://n.news.naver.com/mnews/article/003/0013817590?sid=105</link><description>과학기술정보통신부, 산업통상자원부, 중소벤처기업부가 산업 &lt;b&gt;인공지능&lt;/b&gt;(AI) 전문기업과 제조기업의 AI전환(AX) 사업 참여 환경 개선을 위해 공동 대응한다. 과기정통부와 산업부, 중기부는 올해 주요 AX 사업을 통합... </description><pubDate>Thu, 12 Mar 2026 12:00:00 +0900</pubDate></item><item><title>이재명 대통령 “약속은 지킵니다”…‘국민주권정부’ 성과 알리기 나...</title><originallink>https://www.mk.co.kr/article/11987118</originallink><link>https://n.news.naver.com/mnews/article/009/0005649646?sid=100</link><description>이 대통